# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subata24/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))


In [3]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6874be50f8a466c2cb44d843', 'name': 'Subata24', 'fullname': 'Subata Khan', 'isPro': False, 'avatarUrl': '/avatars/9553ed3ee15a84f25c69dcb4f4f5fb17.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank-colab', 'role': 'fineGrained', 'createdAt': '2026-08-02T12:26:24.583Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6874be50f8a466c2cb44d843', 'type': 'user', 'name': 'Subata24'}, 'permissions': ['repo.content.read']}]}}}}


In [4]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

print(ds)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

Dataset({
    features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
    num_rows: 104
})


In [6]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("FlyRank/internship-warehouse")
print(configs)

['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis**: One row = one content page (client_hash_id + content_hash_id), summarized over one calendar month.

**Table(s)**: fact_content_daily_performance (daily GSC/GA4 facts, grouped to page × month) joined to dim_content for page metadata.

**Time window**: Developing on month=2026-03. month=2026-06 (the _sample file) is a sealed test month, untouched until later.

**Label/proxy**: A decline flag built from this page's gsc_clicks this month vs. its own prior-month trend — a proxy for "needs refresh," not ground truth.

**Deliberate exclusion**: Pages with near-zero impressions in a given month — decline off almost no traffic is noise, not signal..

In [14]:
%pip install -q duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{}');".format(os.environ["HF_TOKEN"]))

# List every file in the dataset so we know exact table/partition names
con.sql("""
SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
""").show(max_rows=50)

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/.gitattributes                                              │
│ hf://datasets/FlyRank/internship-warehouse/README.md                                                   │
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

In [15]:
# Columns for the daily fact table (mid-panel month)
con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").show(max_rows=50)

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gs

In [16]:
# Columns for the content/page dimension table
con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").show(max_rows=50)

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label (derived, not raw)**: Built from gsc_clicks, aggregated to page × month, compared against the same page's prior month(s) → a decline flag/proxy. Only the current month's clicks feed the label — never reused as a feature.

**Features (knowable at decision time — from the prior month or static)**:
- gsc_avg_position (prior month) — ranking strength trend
- sessions_organic (prior month) — traffic trend independent of GSC
- ga4_engaged_sessions (prior month) — engagement signal
- word_count / char_count — content depth
- backlinks — page authority signal
- search_volume, competition — keyword/market context
- content_updated_date → engineered as "days since last update"
- main_intent, content_type — categorical page context

**Context (identifiers, not predictive)**: client_hash_id, content_hash_id, url_hash_id, keyword_hash_id, report_date, month

**Excluded (with why)**:
- optimization_eligible_date — likely leakage; may already encode an editor's decision about this exact question
- is_deleted — deleted pages aren't refresh candidates
- provider_used, model_used — content-ops metadata, not page-performance signal
- current-month gsc_clicks / gsc_impressions / gsc_sum_position / etc. as features — reserved for the label only; using this month's own outcome columns as inputs would leak the answer

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — Grain**: Checked for duplicates at client_hash_id + content_hash_id + report_date on month=2026-03. Result: 9,841,378 total rows, 0 duplicates — confirms one row = one page, one day at this grain.

**Query 2 — Row count and date span**: month=2026-03 contains 9,841,378 rows, 331,437 distinct pages, 55 distinct clients, spanning the full month (2026-03-01 to 2026-03-31) with no gaps at the edges.

**Query 3 — Availability (IS TRUE)**: All 9,841,378 rows belong to clients with GSC connected (client_has_gsc IS TRUE = 100%), but only 3,611,061 rows (~36.7%) have gsc_data_available IS TRUE. Nearly two-thirds of page-days have no usable GSC data for that day — a major availability gap that directly limits the decline label, since a page-day with no GSC data can't be scored for decline.

**Query 1** — Grain check. Prove that client_hash_id + content_hash_id + report_date really is one row (i.e., no duplicates at that grain).

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || report_date) AS duplicate_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ duplicate_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │              0 │
└────────────┴────────────────┘



**Query 2** — row count + date span for your slice

In [18]:
con.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS distinct_pages,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").show()

┌───────────┬────────────────┬──────────────────┬───────────────┬─────────────┐
│ row_count │ distinct_pages │ distinct_clients │ earliest_date │ latest_date │
│   int64   │     int64      │      int64       │     date      │    date     │
├───────────┼────────────────┼──────────────────┼───────────────┼─────────────┤
│   9841378 │         331437 │               55 │ 2026-03-01    │ 2026-03-31  │
└───────────┴────────────────┴──────────────────┴───────────────┴─────────────┘



**Query 3** — availability, filtered with IS TRUE. This checks how many rows actually have usable GSC data (since your label depends on gsc_clicks), using the boolean flag columns we saw earlier (gsc_has_data, gsc_data_available — whichever one is the real boolean flag).

In [19]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS gsc_enabled_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_data_available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┬─────────────────────────┐
│ total_rows │ gsc_enabled_rows │ gsc_data_available_rows │
│   int64    │      int64       │          int64          │
├────────────┼──────────────────┼─────────────────────────┤
│    9841378 │          9841378 │                 3611061 │
└────────────┴──────────────────┴─────────────────────────┘



**Five features, each "available when?"**

- **avg_position_feb** — knowable at decision moment because it's February's average ranking position, fully closed before March begins.
- **sessions_organic_feb** — knowable because it's February's organic session total from GA4, closed before March (available only for clients with GA4 connected — 81,138 rows are null for this reason).
- **engaged_sessions_feb** — same as above: February GA4 data, closed before the decision moment, null where GA4 isn't connected.
- **word_count** — knowable because it's a static content attribute that exists independent of any month's performance (some pages lack this metadata: 50,994 nulls).
- **search_volume** — knowable because it's a keyword-market attribute set before performance is measured, not derived from this page's own outcomes (17,476 nulls where metadata is missing).

In [20]:
label_df = con.sql("""
WITH feb AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_i===--d
),
mar AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_mar
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    mar.client_hash_id,
    mar.content_hash_id,
    feb.clicks_feb,
    mar.clicks_mar,
    CASE WHEN mar.clicks_mar < feb.clicks_feb THEN 1 ELSE 0 END AS declined
FROM mar
JOIN feb ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
""").df()

print(label_df.shape)
label_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(134238, 5)


,client_hash_id,content_hash_id,clicks_feb,clicks_mar,declined
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,7.0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.0,0.0,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,4.0,6.0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,19.0,13.0,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,0.0,1.0,0


In [21]:
features_df = con.sql("""
WITH feb_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS sessions_organic_feb,
        SUM(ga4_engaged_sessions) AS engaged_sessions_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    fm.client_hash_id,
    fm.content_hash_id,
    fm.avg_position_feb,
    fm.sessions_organic_feb,
    fm.engaged_sessions_feb,
    dc.word_count,
    DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-01') AS days_since_update
FROM feb_metrics fm
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' dc
    ON fm.client_hash_id = dc.client_hash_id AND fm.content_hash_id = dc.content_hash_id
""").df()

print(features_df.shape)
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(153559, 7)


,client_hash_id,content_hash_id,avg_position_feb,sessions_organic_feb,engaged_sessions_feb,word_count,days_since_update
0,client_e547b89c05043229,content_1eea820697c3b95a,12.946228,0.0,0.0,2613,-87
1,client_e547b89c05043229,content_9abd8b303f805847,6.495085,3.0,0.0,2992,-113
2,client_e547b89c05043229,content_5f58c55cbfee172a,10.490023,0.0,0.0,2225,-87
3,client_e547b89c05043229,content_6fe390ba3af1e456,38.436254,5.0,1.0,2797,-86
4,client_e547b89c05043229,content_3ad5d2160242b9ca,9.710810,1.0,0.0,2396,-86


In [22]:
features_df = con.sql("""
WITH feb_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS sessions_organic_feb,
        SUM(ga4_engaged_sessions) AS engaged_sessions_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    fm.client_hash_id,
    fm.content_hash_id,
    fm.avg_position_feb,
    fm.sessions_organic_feb,
    fm.engaged_sessions_feb,
    dc.word_count,
    CASE
        WHEN dc.content_updated_date <= DATE '2026-02-28'
        THEN DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-01')
        ELSE NULL
    END AS days_since_update
FROM feb_metrics fm
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' dc
    ON fm.client_hash_id = dc.client_hash_id AND fm.content_hash_id = dc.content_hash_id
""").df()

print(features_df.shape)
features_df.head()
print("Nulls in days_since_update:", features_df['days_since_update'].isna().sum())

(153559, 7)
Nulls in days_since_update: 115800


In [23]:
features_df = con.sql("""
WITH feb_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS sessions_organic_feb,
        SUM(ga4_engaged_sessions) AS engaged_sessions_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    fm.client_hash_id,
    fm.content_hash_id,
    fm.avg_position_feb,
    fm.sessions_organic_feb,
    fm.engaged_sessions_feb,
    dc.word_count,
    dc.search_volume
FROM feb_metrics fm
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' dc
    ON fm.client_hash_id = dc.client_hash_id AND fm.content_hash_id = dc.content_hash_id
""").df()

print(features_df.shape)
features_df.isna().sum()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(153559, 7)


,0
client_hash_id,0
content_hash_id,0
avg_position_feb,0
sessions_organic_feb,81138
engaged_sessions_feb,81138
word_count,50994
search_volume,17476


**The leakage trap**

Honest model (5 real features, all knowable before March closes): **AUC 0.608**.

First leak attempt — adding `clicks_mar` alone: AUC 0.602, essentially unchanged. This didn't leak, because `declined` is a comparison (`clicks_mar < clicks_feb`), not a raw threshold on `clicks_mar` alone — one side of an inequality isn't enough for the model to reconstruct the label.

Second leak attempt — adding `clicks_diff = clicks_mar - clicks_feb`, which directly encodes the label logic: AUC jumped to **1.0**. This column is not a feature, it's the label rewritten — it was deleted immediately after confirming the jump.

**Kept: the honest AUC of 0.608**, using only the five features that were genuinely knowable before the decision moment.

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

merged = label_df.merge(features_df, on=["client_hash_id", "content_hash_id"])
merged = merged.dropna(subset=["avg_position_feb", "word_count", "search_volume"])

X_honest = merged[["avg_position_feb", "sessions_organic_feb", "engaged_sessions_feb", "word_count", "search_volume"]].fillna(0)
y = merged["declined"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC (5 real features only):", honest_score)

Honest AUC (5 real features only): 0.6081207592217026


In [27]:
merged["leaky_clicks_mar"] = merged["clicks_mar"]

X_leaky = merged[["avg_position_feb", "sessions_organic_feb", "engaged_sessions_feb", "word_count", "search_volume", "leaky_clicks_mar"]].fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(X_leaky, y, test_size=0.3, random_state=42)

model_leaky = LogisticRegression(max_iter=1000)
model_leaky.fit(X_train_leak, y_train_leak)
leaky_score = roc_auc_score(y_test_leak, model_leaky.predict_proba(X_test_leak)[:, 1])
print("Leaky AUC (with clicks_mar smuggled in):", leaky_score)
print("Honest AUC was:", honest_score)

Leaky AUC (with clicks_mar smuggled in): 0.601860537600814
Honest AUC was: 0.6081207592217026


In [28]:
merged["leak_clicks_diff"] = merged["clicks_mar"] - merged["clicks_feb"]  # this basically IS the label

X_leaky2 = merged[["avg_position_feb", "sessions_organic_feb", "engaged_sessions_feb", "word_count", "search_volume", "leak_clicks_diff"]].fillna(0)

X_train_leak2, X_test_leak2, y_train_leak2, y_test_leak2 = train_test_split(X_leaky2, y, test_size=0.3, random_state=42)

model_leaky2 = LogisticRegression(max_iter=1000)
model_leaky2.fit(X_train_leak2, y_train_leak2)
leaky_score2 = roc_auc_score(y_test_leak2, model_leaky2.predict_proba(X_test_leak2)[:, 1])
print("Leaky AUC (with clicks_diff smuggled in):", leaky_score2)
print("Honest AUC was:", honest_score)

Leaky AUC (with clicks_diff smuggled in): 1.0
Honest AUC was: 0.6081207592217026


In [29]:
# Deliberate leak removed — honest features only kept going forward
merged = merged.drop(columns=["leak_clicks_diff"], errors="ignore")
print("Final honest AUC (kept):", honest_score)

Final honest AUC (kept): 0.6081207592217026


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**GSC availability gap**: Only ~36.7% of page-days in month=2026-03 have gsc_data_available IS TRUE (Query 3). This means for roughly two-thirds of page-days, there is no way to compute a decline label at all — not because the page didn't decline, but because the day simply wasn't measured. Any model trained on this data is implicitly trained only on the subset of pages/days where GSC happened to report, which is not necessarily representative of all pages.

**Unbalanced per-client history**: dim_clients tracks gsc_data_start and ga4_data_start per client, meaning different clients have different amounts of historical data available. A page's "prior month" comparison may not exist for clients who joined recently — the label can't be computed as a genuine month-over-month trend for every page equally.

**Proxy label, not ground truth**: The decline flag is derived from gsc_clicks trend, not from an editor's actual assessment of content quality or an explicit "needs refresh" tag. A page can show declining clicks for reasons a content refresh won't fix — seasonality, a SERP layout change, or a competitor outranking it — so predictions from this data support editor decisions, they don't replace them.

**Single-month development window**: All contract claims here (grain, counts, availability) were verified on month=2026-03 only. These numbers may not hold for other months — availability rates, page counts, and client mix can shift month to month, so this contract describes 2026-03 specifically, not the full panel.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.